# **Interpolação pela distância de Haversine. Cabreuva/Porto Feliz/Salto para ajustar Sorocaba**

In [19]:
import pandas as pd
import numpy as np
import os
import datetime
from google.colab import drive
from ipywidgets import VBox, Layout, Text, Label, Button, HBox
from IPython.display import display, clear_output

print("### 🔑 CONFIGURAÇÃO E ENTRADA DE PERÍODO ###")

# 1. Montagem do Drive
try:
    drive.mount('/content/drive', force_remount=False)
    print("✔️ Drive montado.")
except:
    print("❌ Falha na montagem do Drive.")

PASTA_DADOS = '/content/drive/MyDrive/Colab Notebooks/Estagio/'


# 2. Input de Período com Botão
print("\n📝 Digite a Data Inicial e a Data Final no formato DD/MM/AAAA e clique em 'Confirmar Período':")

# Campos de entrada de texto para as duas datas
data_inicio_widget = Text(description='Data Inicial (DD/MM/AAAA):', value='01/01/2025', layout=Layout(width='200px'))
data_fim_widget = Text(description='Data Final (DD/MM/AAAA):', value='14/11/2025', layout=Layout(width='200px'))
confirm_button = Button(description="Confirmar Período", button_style='primary', layout=Layout(width='180px'))
output_status = VBox([Label("Status: Aguardando Confirmação")])

# Variável global para armazenar as datas
data_periodo = {'inicio': None, 'fim': None}

def on_button_click(b):
    """Função que valida e armazena o período selecionado ao clicar no botão."""
    global data_periodo

    # Limpa a saída anterior
    output_status.children = [Label("Status: Validando...")]

    data_inicio_str = data_inicio_widget.value
    data_fim_str = data_fim_widget.value

    try:
        # Tenta converter as strings completas no formato esperado
        inicio = datetime.datetime.strptime(data_inicio_str, '%d/%m/%Y').date()
        fim = datetime.datetime.strptime(data_fim_str, '%d/%m/%Y').date()

        if inicio > fim:
             raise ValueError("A Data Inicial não pode ser posterior à Data Final.")

        data_periodo['inicio'] = inicio
        data_periodo['fim'] = fim

        output_status.children = [
            Label(f"✅ Período selecionado: {inicio.strftime('%d/%m/%Y')} a {fim.strftime('%d/%m/%Y')}."),
            Label("Agora, execute a Etapa 2.")
        ]
    except ValueError as e:
        data_periodo['inicio'] = None
        data_periodo['fim'] = None
        output_status.children = [
            Label(f"❌ ERRO: Data(s) inválida(s) ou ordem incorreta. Detalhe: {e}")
        ]

# Associa a função ao evento de clique
confirm_button.on_click(on_button_click)

# Exibe os inputs e o botão
display(VBox([
    HBox([data_inicio_widget, data_fim_widget]),
    HBox([confirm_button]),
    output_status
]))

### 🔑 CONFIGURAÇÃO E ENTRADA DE PERÍODO ###
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✔️ Drive montado.

📝 Digite a Data Inicial e a Data Final no formato DD/MM/AAAA e clique em 'Confirmar Período':


In [20]:
# --- 1. Importações e Definições ---
from google.colab import files
import pandas as pd
import numpy as np
import datetime
import os

# Verifica a disponibilidade da variável global 'data_periodo' do input
try:
    if 'data_periodo' not in globals() or data_periodo['inicio'] is None or data_periodo['fim'] is None:
        raise Exception("Execute a Etapa 1 e clique no botão 'Confirmar Período' para selecionar um período válido.")
except NameError:
    print("❌ ERRO: Variáveis globais não encontradas. Certifique-se de que ambas as células foram executadas na ordem correta.")
    exit()

# Função para download
def exportar_e_baixar_csv(df, filename, cols_to_format=[]):
    """Salva o DataFrame em CSV formatado e inicia o download."""

    # Formata as colunas numéricas para usar vírgula (,) como separador decimal
    for col in cols_to_format:
        if col in df.columns:
            # Garante que NaN (float) seja tratado antes da formatação. Valores não numéricos serão ignorados
            df[col] = pd.to_numeric(df[col], errors='coerce').round(2).astype(str).str.replace('.', ',', regex=False).replace('nan', '', regex=False)

    try:
        df.to_csv(filename, sep=';', index=False, encoding='utf-8')
        print(f"✔️ Arquivo **{filename}** criado e pronto para download.")
        files.download(filename)
    except Exception as e:
        print(f"❌ ERRO ao salvar ou baixar o arquivo {filename}: {e}")

# Função para extrair nome da estação corretamente (CORREÇÃO APLICADA AQUI)
def extract_station_name(col_name, var_names):
    """Extrai o nome da estação de colunas formatadas como 'Estacao_Variavel'."""
    for var in var_names:
        var_suffix = f'_{var}'
        if col_name.endswith(var_suffix):
            # Retorna a parte anterior à variável, que é o nome da estação
            return col_name[:-len(var_suffix)]
    return None

# --- 2. Definições de Funções (Interpolação) ---
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371
    lat1_rad, lon1_rad = np.radians(lat1), np.radians(lon1)
    lat2_rad, lon2_rad = np.radians(lat2), np.radians(lon2)
    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad
    a = np.sin(dlat / 2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def idw_interpolation_com_contagem(row_of_values, dists_series, p=2):
    """Retorna o valor IDW e a contagem de vizinhos usados."""
    vizinhos = row_of_values.dropna()
    if vizinhos.empty: return np.nan, 0

    dists_vizinhos = dists_series.loc[vizinhos.index]
    dists_validas = dists_vizinhos[dists_vizinhos > 0]
    vizinhos_validos = vizinhos.loc[dists_validas.index]

    contagem_vizinhos = len(vizinhos_validos)

    if vizinhos_validos.empty: return np.nan, 0

    pesos = 1 / (dists_validas ** p)
    idw_value = (vizinhos_validos * pesos).sum() / pesos.sum()
    return idw_value, contagem_vizinhos

# --- 3. Setup de Estações e Variáveis ---
ALVO = 'Sorocaba'
VIZINHOS = ['Salto', 'Cabreuva', 'Porto_Feliz']
COLUNA_DATA = 'Data'
COLUNA_HORA = 'Hora (UTC)'
COLUNA_VALOR_TEMP = 'Temp. Ins. (C)'
COLUNA_VALOR_UMI = 'Umi. Ins. (%)'
VARS_TO_INTERPOLATE = [COLUNA_VALOR_TEMP, COLUNA_VALOR_UMI]

NOVA_COLUNA_DATAHORA = 'DataHora'
PASTA_DADOS = '/content/drive/MyDrive/Colab Notebooks/Estagio/'

coordenadas = {
    'Sorocaba': [-23.500, -47.450], 'Salto': [-23.210, -47.280],
    'Cabreuva': [-23.300, -47.160], 'Porto_Feliz': [-23.250, -47.530]
}
distancias = {}
lat_alvo, lon_alvo = coordenadas[ALVO]
for vizinho in VIZINHOS:
    dist = haversine_distance(lat_alvo, lon_alvo, coordenadas[vizinho][0], coordenadas[vizinho][1])
    distancias[vizinho] = dist
distancias_series = pd.Series(distancias)
print(f"Distâncias (em km) de Sorocaba para Vizinhos:\n{distancias_series.to_string()}\n")


# --- 4. Carregamento de Dados ---
dfs_interp = {} # DF para Interpolação (apenas Temp e Umi, renomeadas)
df_sorocaba_bruto_full = pd.DataFrame() # DF Bruto Completo (Arquivo 2)

for estacao in [ALVO] + VIZINHOS:
    arquivo = f'{PASTA_DADOS}{estacao}.csv'
    try:
        df = pd.read_csv(arquivo, sep=';', encoding='utf-8-sig', decimal=',', quotechar='"')
        df.columns = [col.strip() for col in df.columns]

        df[NOVA_COLUNA_DATAHORA] = pd.to_datetime(
            df[COLUNA_DATA] + ' ' + df[COLUNA_HORA].astype(str).str.zfill(4),
            format='%d/%m/%Y %H%M'
        )
        df = df.set_index(NOVA_COLUNA_DATAHORA)

        if estacao == ALVO:
            df_sorocaba_bruto_full = df.copy()

        # Prepara DF para Interpolação (renomeando para evitar conflitos de coluna)
        df_temp_umi = df[VARS_TO_INTERPOLATE].copy()
        df_temp_umi = df_temp_umi.rename(columns={
            COLUNA_VALOR_TEMP: f'{estacao}_{COLUNA_VALOR_TEMP}',
            COLUNA_VALOR_UMI: f'{estacao}_{COLUNA_VALOR_UMI}'
        })
        dfs_interp[estacao] = df_temp_umi

    except FileNotFoundError:
        pass
    except Exception:
         pass

if ALVO not in dfs_interp:
    print("❌ ERRO FINAL: O arquivo ALVO (Sorocaba.csv) não foi carregado. Finalizando.")
    exit()

# Combinação de todas as colunas necessárias para interpolação
df_completo_multi = pd.concat([dfs_interp[s] for s in [ALVO] + VIZINHOS if s in dfs_interp], axis=1).sort_index()
df_final_interpolated = df_completo_multi.copy() # DF que receberá os valores interpolados

# --- 5. Interpolação de Múltiplas Variáveis e Rastreamento ---
all_interpolated_indices = set()
interpolated_records = [] # Para o Arquivo 1 (Interp Apenas)
sucesso_temp = 0
sucesso_umi = 0

for VAR in VARS_TO_INTERPOLATE:
    VAR_SUCESSO_COUNT = 0
    ALVO_COL_NAME = f'{ALVO}_{VAR}'
    NEIGHBOR_COLS_NAMES = [f'{v}_{VAR}' for v in VIZINHOS]

    # Isola a coluna alvo e as colunas dos vizinhos para esta variável
    df_var = df_final_interpolated[[ALVO_COL_NAME] + NEIGHBOR_COLS_NAMES].copy()
    gaps_var = df_var[df_var[ALVO_COL_NAME].isna()]

    if not gaps_var.empty:
        for datahora, row in gaps_var.iterrows():

            valores_vizinhos_raw = row[NEIGHBOR_COLS_NAMES].dropna()

            if not valores_vizinhos_raw.empty:
                # Mapeia de volta para o nome da estação para usar a série de distâncias
                vizinhos_usados_nomes = [extract_station_name(c, VARS_TO_INTERPOLATE) for c in valores_vizinhos_raw.index]
                vizinhos_usados_nomes = [n for n in vizinhos_usados_nomes if n in distancias_series.index] # Filtra nomes válidos

                if not vizinhos_usados_nomes:
                    continue

                dists_filtradas = distancias_series.loc[vizinhos_usados_nomes]

                # Cria uma Série temporária para interpolação: valores indexados pelo nome da estação
                # Precisa mapear os valores de volta para os nomes de vizinhos usados para as distâncias
                valores_filtrados = valores_vizinhos_raw.loc[[f'{n}_{VAR}' for n in vizinhos_usados_nomes]]
                temp_series = pd.Series(valores_filtrados.values, index=vizinhos_usados_nomes)

                valor_interpolado, contagem_vizinhos = idw_interpolation_com_contagem(temp_series, dists_filtradas, p=2)

                if pd.notna(valor_interpolado):
                    # 1. Atualiza o DF final com o valor interpolado
                    df_final_interpolated.loc[datahora, ALVO_COL_NAME] = valor_interpolado

                    # 2. Rastreia o índice
                    all_interpolated_indices.add(datahora)

                    # 3. Armazena o registro detalhado para o Arquivo 1 (Interp Apenas)
                    record = {'DataHora': datahora}
                    record[f'Interp_{VAR}'] = valor_interpolado
                    record[f'Vizinhos_Usados_{VAR}'] = contagem_vizinhos

                    interpolated_records.append(record)
                    VAR_SUCESSO_COUNT += 1

    if VAR == COLUNA_VALOR_TEMP:
        sucesso_temp = VAR_SUCESSO_COUNT
    elif VAR == COLUNA_VALOR_UMI:
        sucesso_umi = VAR_SUCESSO_COUNT

print(f"\n✅ Gaps preenchidos com sucesso via IDW:")
print(f"   - {sucesso_temp} registros de Temperatura.")
print(f"   - {sucesso_umi} registros de Umidade.")

# --- 6. Filtro de Período ---
data_inicio_filtro = data_periodo['inicio']
data_fim_filtro = data_periodo['fim']
timestamp_inicio = pd.Timestamp(data_inicio_filtro)
timestamp_fim = pd.Timestamp(data_fim_filtro) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)

df_final_interpolated_filtrado = df_final_interpolated.loc[timestamp_inicio:timestamp_fim].copy()
df_sorocaba_bruto_full_filtrado = df_sorocaba_bruto_full.loc[timestamp_inicio:timestamp_fim].copy()

print(f"\n✅ Dados filtrados para o período: {data_inicio_filtro.strftime('%d/%m/%Y')} a {data_fim_filtro.strftime('%d/%m/%Y')}")
indices_interpolados_filtrados = [idx for idx in all_interpolated_indices if timestamp_inicio <= idx <= timestamp_fim]


# --- 7. Download do ARQUIVO 1: APENAS INTERPOLADOS (Temp e Umi) ---
OUTPUT_FILENAME_INTERP_APENAS = f'Sorocaba_Interp_Apenas_{data_inicio_filtro.strftime("%Y%m%d")}_a_{data_fim_filtro.strftime("%Y%m%d")}.csv'

if indices_interpolados_filtrados:

    # 1. Processa a lista de records para um DF estruturado
    df_interp_apenas_raw = pd.DataFrame(interpolated_records)

    # 2. Pivot/Aggregate os valores interpolados e contagens por DataHora
    cols_agg = {}
    for VAR in VARS_TO_INTERPOLATE:
        # Usa max() em vez de first() para garantir que pegue o valor mesmo se for Nan
        cols_agg[f'Interp_{VAR}'] = 'max'
        cols_agg[f'Vizinhos_Usados_{VAR}'] = 'max'

    df_interp_pivoted = df_interp_apenas_raw.groupby('DataHora').agg(cols_agg)

    # 3. Filtra para o período e junta os dados brutos dos vizinhos (Temp e Umi)
    df1_export = df_interp_pivoted.loc[indices_interpolados_filtrados].copy()

    # Junta com os dados brutos dos vizinhos (todos no df_completo_multi)
    vizinhos_cols_interp = [c for c in df_completo_multi.columns if c.split('_')[0] in VIZINHOS]
    df1_export = df1_export.join(df_completo_multi[vizinhos_cols_interp], how='left')

    # 4. Renomeia e formata
    df1_export = df1_export.rename(columns={
        f'Interp_{COLUNA_VALOR_TEMP}': COLUNA_VALOR_TEMP,
        f'Interp_{COLUNA_VALOR_UMI}': COLUNA_VALOR_UMI,
        f'Vizinhos_Usados_{COLUNA_VALOR_TEMP}': 'Vizinhos_Usados_Temp (0-3)',
        f'Vizinhos_Usados_{COLUNA_VALOR_UMI}': 'Vizinhos_Usados_Umi (0-3)',
    }).reset_index()

    df1_export[COLUNA_DATA] = df1_export['DataHora'].dt.strftime('%d/%m/%Y')
    df1_export[COLUNA_HORA] = df1_export['DataHora'].dt.strftime('%H%M')
    df1_export = df1_export.drop(columns=['DataHora'])

    # Ordem das colunas
    cols_to_keep = [COLUNA_DATA, COLUNA_HORA, COLUNA_VALOR_TEMP, 'Vizinhos_Usados_Temp (0-3)', COLUNA_VALOR_UMI, 'Vizinhos_Usados_Umi (0-3)']
    cols_to_keep += [c for c in df1_export.columns if c not in cols_to_keep]
    df1_export = df1_export[cols_to_keep]

    cols_to_format1 = [c for c in df1_export.columns if c not in [COLUNA_DATA, COLUNA_HORA, 'Vizinhos_Usados_Temp (0-3)', 'Vizinhos_Usados_Umi (0-3)']]

    print("\n--- INICIANDO DOWNLOADS ---")
    exportar_e_baixar_csv(df1_export, OUTPUT_FILENAME_INTERP_APENAS, cols_to_format=cols_to_format1)
else:
    df_vazio = pd.DataFrame([["Não foram encontrados dados interpolados de Temperatura ou Umidade no período selecionado."]], columns=['Status'])
    exportar_e_baixar_csv(df_vazio, OUTPUT_FILENAME_INTERP_APENAS)


# --- 8. Download do ARQUIVO 2: BRUTO COMPLETO DE SOROCABA (APENAS ORIGINAIS) ---
OUTPUT_FILENAME_BRUTO_COMPLETO = f'Sorocaba_Completo_Bruto_{data_inicio_filtro.strftime("%Y%m%d")}_a_{data_fim_filtro.strftime("%Y%m%d")}.csv'

if not df_sorocaba_bruto_full_filtrado.empty:
    df2_export = df_sorocaba_bruto_full_filtrado.reset_index().drop(columns=['DataHora'])

    cols_originais = [c for c in df2_export.columns if c not in [COLUNA_DATA, COLUNA_HORA]]
    df2_export = df2_export[[COLUNA_DATA, COLUNA_HORA] + cols_originais]

    cols_to_format2 = [c for c in df2_export.columns if c not in [COLUNA_DATA, COLUNA_HORA, 'Dir. Vento (m/s)']]

    exportar_e_baixar_csv(df2_export, OUTPUT_FILENAME_BRUTO_COMPLETO, cols_to_format=cols_to_format2)
else:
    print(f"❌ Não foi possível gerar o arquivo bruto completo de Sorocaba para o período.")


# --- 9. Download do ARQUIVO 3: COMPLETO COM STATUS (CONTÉM VALORES INTERPOLADOS) ---
OUTPUT_FILENAME_COMPLETO_STATUS = f'Sorocaba_Completo_Status_{data_inicio_filtro.strftime("%Y%m%d")}_a_{data_fim_filtro.strftime("%Y%m%d")}.csv'

df_export_3 = df_final_interpolated_filtrado.copy()

# 1. Cria a coluna Status_Interp (se T ou U foi interpolado)
df_export_3['Status_Interp (Temp/Umi)'] = ''
df_export_3.loc[indices_interpolados_filtrados, 'Status_Interp (Temp/Umi)'] = 'Interpolado'

# 2. Renomeia e formata
df_export_3 = df_export_3.reset_index()
df_export_3[COLUNA_DATA] = df_export_3[NOVA_COLUNA_DATAHORA].dt.strftime('%d/%m/%Y')
df_export_3[COLUNA_HORA] = df_export_3[NOVA_COLUNA_DATAHORA].dt.strftime('%H%M')

df_export_3 = df_export_3.drop(columns=[NOVA_COLUNA_DATAHORA])

# 3. Reordena colunas: Data, Hora, [Sorocaba Temp/Umi], Status, Vizinhos
cols_target = [f'{ALVO}_{VAR}' for VAR in VARS_TO_INTERPOLATE]
cols_vizinhos = [c for c in df_export_3.columns if any(v in c for v in VIZINHOS)]

cols_ordered3 = [COLUNA_DATA, COLUNA_HORA] + cols_target + ['Status_Interp (Temp/Umi)'] + cols_vizinhos
df_export_3 = df_export_3[cols_ordered3]

# 4. Remove o prefixo 'Sorocaba_' e 'Vizinho_' para a saída
df_export_3.columns = [c.replace(f'{ALVO}_', '') if c.startswith(f'{ALVO}_') else c for c in df_export_3.columns]
df_export_3.columns = [c.replace(f'_', ' ') if c not in [COLUNA_DATA, COLUNA_HORA, 'Status_Interp (Temp/Umi)'] else c for c in df_export_3.columns]

cols_to_format3 = [c for c in df_export_3.columns if c not in [COLUNA_DATA, COLUNA_HORA, 'Status_Interp (Temp/Umi)']]
exportar_e_baixar_csv(df_export_3, OUTPUT_FILENAME_COMPLETO_STATUS, cols_to_format=cols_to_format3)


print("\nTODOS OS TRÊS DOWNLOADS FORAM INICIADOS. Verifique sua pasta de downloads.")

Distâncias (em km) de Sorocaba para Vizinhos:
Salto          36.619809
Cabreuva       37.018905
Porto_Feliz    28.973176


✅ Gaps preenchidos com sucesso via IDW:
   - 696 registros de Temperatura.
   - 696 registros de Umidade.

✅ Dados filtrados para o período: 14/10/2025 a 23/10/2025

--- INICIANDO DOWNLOADS ---
✔️ Arquivo **Sorocaba_Interp_Apenas_20251014_a_20251023.csv** criado e pronto para download.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✔️ Arquivo **Sorocaba_Completo_Bruto_20251014_a_20251023.csv** criado e pronto para download.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✔️ Arquivo **Sorocaba_Completo_Status_20251014_a_20251023.csv** criado e pronto para download.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


TODOS OS TRÊS DOWNLOADS FORAM INICIADOS. Verifique sua pasta de downloads.
